
# Task 2: Feature Engineering (NumPy and Pandas)

In [1]:
import logging

log_file = r"C:\Users\symev\OneDrive\Training\Emeritus\capstone project\emeritus captone project\part2_python\feature_engineering.log"

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

logger.handlers.clear()

file_handler = logging.FileHandler(
    log_file,
    mode="w",
    encoding="utf-8"
)

formatter = logging.Formatter(
    "{asctime} - {levelname} - {name} - {message}",
    style="{",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Test that logging is working
logger.info("Logging started successfully.")

In [2]:
import pandas as pd
import numpy as np

In [3]:
try:
    # Load the raw CSV file
    df = pd.read_csv('C:\\Users\\symev\\OneDrive\\Training\\Emeritus\\capstone project\\emeritus captone project\\Metro_Interstate_Traffic_Volume_Cleaned.csv')

    # Get number of rows and columns
    rows, columns = df.shape

    # Log successful data loading
    logger.info(
        "Raw CSV file loaded successfully: %d rows and %d columns.",
        rows,
        columns
    )

    # Display confirmation and first few rows
    print(f"\nFile loaded successfully!")
    print(f"Rows: {rows}")
    print(f"Columns: {columns}")
    print("\nFirst 5 rows:")
    print(df.head())

except FileNotFoundError:
    logger.error(f"File not found: '{file_path}'.")

except pd.errors.EmptyDataError:
    logger.error("The CSV file is empty.")

except pd.errors.ParserError:
    logger.error("The CSV file could not be parsed. Please check its format.")

except PermissionError:
    logger.error(f"Permission denied when trying to access '{file_path}'.")

except OSError as e:
    logger.error(f"Error reading the file: {e}")


File loaded successfully!
Rows: 48176
Columns: 9

First 5 rows:
  holiday    temp  rain_1h  snow_1h  clouds_all weather_main  \
0    None  288.28      0.0      0.0          40       Clouds   
1    None  289.36      0.0      0.0          75       Clouds   
2    None  289.58      0.0      0.0          90       Clouds   
3    None  290.13      0.0      0.0          90       Clouds   
4    None  291.14      0.0      0.0          75       Clouds   

  weather_description            date_time  traffic_volume  
0    Scattered Clouds  2012-10-02 09:00:00            5545  
1       Broken Clouds  2012-10-02 10:00:00            4516  
2     Overcast Clouds  2012-10-02 11:00:00            4767  
3     Overcast Clouds  2012-10-02 12:00:00            5026  
4       Broken Clouds  2012-10-02 13:00:00            4918  


In [4]:
# Parse date_time as datetime
df['date_time'] = pd.to_datetime(
    df['date_time'],
    errors='coerce'
)

In [5]:
# Log dataset shape before feature engineering
rows_before, columns_before = df.shape

logger.info(
    "Dataset shape before feature engineering: %d rows and %d columns.",
    rows_before,
    columns_before
)

print(f"Dataset shape before feature engineering: {df.shape}")

Dataset shape before feature engineering: (48176, 9)


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48176 entries, 0 to 48175
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              48176 non-null  object        
 1   temp                 48176 non-null  float64       
 2   rain_1h              48176 non-null  float64       
 3   snow_1h              48176 non-null  float64       
 4   clouds_all           48176 non-null  int64         
 5   weather_main         48176 non-null  object        
 6   weather_description  48176 non-null  object        
 7   date_time            48176 non-null  datetime64[ns]
 8   traffic_volume       48176 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(3)
memory usage: 3.3+ MB


In [7]:
#Create time features 

# Hour of the day: 0-23
df['hour'] = df['date_time'].dt.hour

# Day of week: Monday = 0, Sunday = 6
df['day_of_week'] = df['date_time'].dt.dayofweek

# Weekend indicator: 1 = Saturday/Sunday, 0 = weekday
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# Cyclical encoding of hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)

df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Cyclical encoding of hour
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)

df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

In [8]:
# create holiday flag
df['holiday_flag'] = (df['holiday'] != 'None').astype(int)


In [9]:
#create weather features using one-hot encoding
weather_encoded = pd.get_dummies(
    df['weather_main'],
    prefix='weather',
    dtype=int
)

# Add encoded variables to the dataframe
df = pd.concat([df, weather_encoded], axis=1)


In [10]:
#create weather indicator for clear and not clear weater 

# Clear weather indicator
df['weather_is_clear'] = (df['weather_main'] == 'Clear').astype(int)

# Not clear weather indicator
df['weather_is_not_clear'] = (df['weather_main'] != 'Clear').astype(int)

In [11]:
# normalised/scaled versions of numerical features - temp and traffic volume
from sklearn.preprocessing import MinMaxScaler

# Create scaler
scaler = MinMaxScaler()

# Apply Min-Max scaling
df[['temp_scaled', 'traffic_volume_scaled']] = scaler.fit_transform(
    df[['temp', 'traffic_volume']]
)

In [12]:
# create traffic congestion categories based on data 

# Calculate percentile thresholds
low_threshold = df['traffic_volume'].quantile(0.33)
high_threshold = df['traffic_volume'].quantile(0.67)

# Log intermediate threshold values at DEBUG level
logger.debug(
    "Congestion thresholds calculated: "
    "33rd percentile = %.2f, 67th percentile = %.2f",
    low_threshold,
    high_threshold
)


# Create congestion category
def classify_congestion(volume):
    if volume <= low_threshold:
        return 'Low'
    elif volume <= high_threshold:
        return 'Medium'
    else:
        return 'High'


df['congestion_category'] = df['traffic_volume'].apply(
    classify_congestion
)


# Display final category distribution
print("\nCongestion category distribution:")
print(df['congestion_category'].value_counts())


Congestion category distribution:
Medium    16376
Low       15908
High      15892
Name: congestion_category, dtype: int64


In [13]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,hour,...,weather_Rain,weather_Smoke,weather_Snow,weather_Squall,weather_Thunderstorm,weather_is_clear,weather_is_not_clear,temp_scaled,traffic_volume_scaled,congestion_category
0,None,288.28,0.0,0.0,40,Clouds,Scattered Clouds,2012-10-02 09:00:00,5545,9,...,0,0,0,0,0,0,1,0.673215,0.761676,High
1,None,289.36,0.0,0.0,75,Clouds,Broken Clouds,2012-10-02 10:00:00,4516,10,...,0,0,0,0,0,0,1,0.689412,0.620330,Medium
2,None,289.58,0.0,0.0,90,Clouds,Overcast Clouds,2012-10-02 11:00:00,4767,11,...,0,0,0,0,0,0,1,0.692711,0.654808,High
3,None,290.13,0.0,0.0,90,Clouds,Overcast Clouds,2012-10-02 12:00:00,5026,12,...,0,0,0,0,0,0,1,0.700960,0.690385,High
4,None,291.14,0.0,0.0,75,Clouds,Broken Clouds,2012-10-02 13:00:00,4918,13,...,0,0,0,0,0,0,1,0.716107,0.675549,High


In [14]:
# Log dataset shape after feature engineering
rows_after, columns_after = df.shape

logger.info(
    "Dataset shape after feature engineering: %d rows and %d columns.",
    rows_after,
    columns_after
)
print(f"Dataset shape after feature engineering: {df.shape}")

Dataset shape after feature engineering: (48176, 33)


In [15]:
# Export cleaned data to CSV
output_file = r"C:\Users\symev\OneDrive\Training\Emeritus\capstone project\emeritus captone project\Metro_Interstate_Traffic_Volume_Cleaned_FeatureEngineering.csv"

try:
    df.to_csv(output_file, index=False)

    logger.info(
        "Cleaned data exported successfully to: %s",
        output_file
    )

    print(f"Cleaned data exported successfully to:\n{output_file}")

except OSError as e:
    logger.error(
        "Failed to export cleaned data: %s",
        e
    )
    print(f"Error exporting cleaned data: {e}")

Cleaned data exported successfully to:
C:\Users\symev\OneDrive\Training\Emeritus\capstone project\emeritus captone project\Metro_Interstate_Traffic_Volume_Cleaned_FeatureEngineering.csv


In [16]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,hour,...,weather_Rain,weather_Smoke,weather_Snow,weather_Squall,weather_Thunderstorm,weather_is_clear,weather_is_not_clear,temp_scaled,traffic_volume_scaled,congestion_category
0,None,288.28,0.0,0.0,40,Clouds,Scattered Clouds,2012-10-02 09:00:00,5545,9,...,0,0,0,0,0,0,1,0.673215,0.761676,High
1,None,289.36,0.0,0.0,75,Clouds,Broken Clouds,2012-10-02 10:00:00,4516,10,...,0,0,0,0,0,0,1,0.689412,0.620330,Medium
2,None,289.58,0.0,0.0,90,Clouds,Overcast Clouds,2012-10-02 11:00:00,4767,11,...,0,0,0,0,0,0,1,0.692711,0.654808,High
3,None,290.13,0.0,0.0,90,Clouds,Overcast Clouds,2012-10-02 12:00:00,5026,12,...,0,0,0,0,0,0,1,0.700960,0.690385,High
4,None,291.14,0.0,0.0,75,Clouds,Broken Clouds,2012-10-02 13:00:00,4918,13,...,0,0,0,0,0,0,1,0.716107,0.675549,High
